# Full segmentation across an AVITI24 cytoprofiling run

After you select a model with the tile evaluation notebook, run this notebook to apply your selected model to every tile in the run and write the segmentation masks Cells2Stats expects.

This notebook is the companion to the [Custom segmentation tutorial](https://docs.elembio.io/docs/tutorials/cytoprofiling/custom-segmentation/). Read the tutorial first for the full context: run-type identification, tile evaluation, and post-segmentation Cells2Stats re-run.


## Prerequisites

Confirm the following before running this notebook:

- You identified your run type. See [Identify run type](https://docs.elembio.io/docs/tutorials/cytoprofiling/custom-segmentation/#identify-run-type).
- You recorded the best model name from the tile evaluation. See [Interpret results and choose a model](https://docs.elembio.io/docs/tutorials/cytoprofiling/custom-segmentation/#interpret-results).
- You installed the cytoprofiling package and know your `segmentationModels/` directory path.
- Your `cytoprofiling-seg` environment is active and selected as this notebook's kernel (top menu: **Kernel → Change kernel → Cytoprofiling Segmentation**).


## Step 1 — Import packages and detect GPU

Cellpose loads the segmentation models, scikit-image reads and writes the tile TIFFs, and the warning filters silence two benign messages from PyTorch and scikit-image during inference. GPU availability is auto-detected; if a GPU is found, all segmentation calls will use it.


In [ ]:
from cellpose import core, models, transforms
import skimage
import os
import numpy as np
import json

import warnings
warnings.filterwarnings("ignore", message=".*weights_only=False.*")
warnings.filterwarnings("ignore", message=".*low contrast image.*")

use_gpu = core.use_gpu()
if use_gpu:
    print("GPU detected — segmentation will use GPU acceleration.")
else:
    print("No GPU detected — segmentation will run on CPU (expect longer runtimes).")


## Step 2 — Provide Input and Output Paths

- `run_directory` should point to a Teton run folder with `Projection/` images (Cell-Membrane, Nucleus, Actin) and `RunParameters.json`. These files determine well layout, kit type, and available tiles.
- `output_location` is the root folder for segmentation outputs. The notebook writes each well's masks under `output_location/Well{well}/` so the directory can be consumed directly by Cells2Stats (the downstream analysis pipeline).
- `model_dir` should point to the `segmentationModels/` folder inside your local clone of the cytoprofiling package (for example, `cytoprofiling/src/segmentationModels`). The notebook resolves every model name in STEP C of the next cell against this directory.


In [ ]:
# Edit all three paths before running the rest of the notebook.

# Path to your AVITI24 run output folder
run_directory   = r"/path/to/your/Run/Output/Folder"

# Where to write the segmentation mask outputs (must be a different folder)
output_location = r"/path/to/your/Run/Output/Folder/Segmentation_Output"

# Path to segmentation models
model_dir       = r"/path/to/cytoprofiling/src/segmentationModels"

## Step 3 — Configure cell types and models

The notebook auto-initializes well-level cell type and diameter assignments from `RunManifest.csv` in `run_directory`. Review the loaded values printed below the cell and make adjustments in the override block before running Step 3.

- **STEP A**: load `cell_types` and `cell_diameters` from the run manifest, with optional per-well overrides.
- **STEP B**: select the segmentation model file used for each cell type. Use `_3ch` for Teton/Teton Atlas runs and `_2ch` for Cell Paint only runs.


In [ ]:
# ───────────────────────────────────────────────────────────────────────
# STEP A — Load cell_types and cell_diameters from RunManifest.csv
# ───────────────────────────────────────────────────────────────────────
# Source of truth is the [Wells] section of RunManifest.csv at run_directory.
# Manifest CellType values are agnostic to spaces, hyphens, and case
# (see https://docs.elembio.io/docs/run-manifest/wells-section/).
# If RunManifest.csv is missing, the well list is read from
# RunParameters.json and every well defaults to OTHER.
import csv
import re

# Map each normalized manifest CellType to the matching STEP B key.
# Update this dict if you add a new model to STEP B with a name that
# does not normalize directly to its key (for example, SH-SY5Y).
MANIFEST_TO_MODEL_KEY = {
    "HELA":       "HELA",
    "HUVEC":      "HUVEC",
    "MCF7":       "MCF7",
    "PC3":        "PC3",
    "HEK293":     "HEK293",
    "JURKAT":     "JURKAT",
    "HEPG2":      "HEPG2",
    "HCT116":     "HCT116",
    "PBMC":       "PBMC",
    "SHSY5Y":     "SH-SY5Y",
    "FIBROBLAST": "FIBROBLAST",
    "IPSC":       "IPSC",
}


def _norm(value: str) -> str:
    return re.sub(r"[\s-]", "", value).upper().strip()


def _iter_manifest_wells(path: str):
    """Yield dicts of {WellLocation, CellType, CellDiameter, ...} from
    the [Wells] section of an Element Biosciences run manifest CSV."""
    in_wells = False
    header = None
    with open(path, newline="") as f:
        for raw in f:
            line = raw.rstrip("\r\n")
            stripped = line.strip()
            if stripped.startswith("[") and stripped.endswith("]"):
                in_wells = (stripped == "[Wells]")
                header = None
                continue
            if not in_wells or not stripped:
                continue
            row = next(csv.reader([line]))
            row = [c.strip() for c in row]
            if header is None:
                header = [c for c in row if c]
                continue
            padded = row + [""] * (len(header) - len(row))
            yield dict(zip(header, padded))


cell_types = {}
cell_diameters = {}

manifest_path = os.path.join(run_directory, "RunManifest.csv")
if os.path.exists(manifest_path):
    for record in _iter_manifest_wells(manifest_path):
        well = record.get("WellLocation", "").strip()
        if not well:
            continue
        manifest_type = record.get("CellType", "").strip()
        ctype = MANIFEST_TO_MODEL_KEY.get(_norm(manifest_type), "OTHER")
        cell_types[well] = ctype
        diameter_raw = record.get("CellDiameter", "").strip()
        if ctype == "OTHER" and diameter_raw:
            try:
                cell_diameters[well] = int(float(diameter_raw))
            except ValueError:
                pass
    print(f"Loaded {len(cell_types)} wells from RunManifest.csv")
else:
    with open(os.path.join(run_directory, "RunParameters.json")) as f:
        rp = json.load(f)
    for w in rp.get("Wells", []):
        cell_types[w["WellLocation"]] = "OTHER"
    print(
        f"RunManifest.csv not found. Loaded {len(cell_types)} wells from "
        "RunParameters.json (all set to OTHER)."
    )

# ───────────────────────────────────────────────────────────────────────
# STEP B — OVERRIDE RUN MANIFEST LOADED MODELS
# ───────────────────────────────────────────────────────────────────────
# Uncomment and edit the entries below if a well's manifest entry is
# wrong or you want to use a different model for a specific well.
# Values set here win over anything loaded from the manifest in STEP A.
# Diameters are only consulted for wells whose cell type is OTHER.
cell_types.update({
     "A1": "HELA",
    # "A2": "HELA",
    # "B1": "HELA",
    # "B2": "HELA",
     "C1": "HELA",
    # "C2": "HELA",
    # "D1": "HELA",
    # "D2": "HELA",
    # "E1": "HELA",
    # "E2": "HELA",
    # "F1": "HELA",
    # "F2": "HELA",
})
cell_diameters.update({
    # "A1": 40,
    # "A2": 40,
    # "B1": 40,
    # "B2": 40,
    # "C1": 40,
    # "C2": 40,
    # "D1": 40,
    # "D2": 40,
    # "E1": 40,
    # "E2": 40,
    # "F1": 40,
    # "F2": 40,
})

print("cell_types     :", json.dumps(cell_types, indent=2))
print("cell_diameters :", json.dumps(cell_diameters, indent=2))


# ───────────────────────────────────────────────────────────────────────
# STEP C — Select your segmentation models
# ───────────────────────────────────────────────────────────────────────
# Each key must match a value in cell_types from STEP A.
# Model suffix must match your run type:
#   _3ch → Teton or Teton Atlas runs (Cell-Membrane + Nucleus + Actin)
#   _2ch → Cell Paint only runs (Cell-Membrane + Nucleus only)
element_models = {
    "OTHER":      "20240905_general_15diam_3ch",   # fallback for unlisted cell types
    "HELA":       "20240905_hela_15diam_3ch",
    "HUVEC":      "20240905_huvec_15diam_3ch",
    "MCF7":       "20240924_mcf7_15diam_3ch",
    "PC3":        "20240924_pc3_15diam_3ch",
    "HEK293":     "20240924_hek293_15diam_3ch",
    "JURKAT":     "20240924_jurkat_15diam_3ch",
    "HEPG2":      "20240924_hepg2_15diam_3ch",
    "HCT116":     "20240924_hct116_15diam_3ch",
    "PBMC":       "20250127_pbmc_15diam_3ch",
    "SH-SY5Y":    "20250127_sh-sy5y_15diam_3ch",
    "FIBROBLAST": "20260403_fibroblast_15diam_3ch",
    "IPSC":       "20260402_ipsc_15diam_3ch",
}

# Nuclear model — same for all runs and cell types
nuclear_model = "20250212_cellpose_nuc_8diam"


## Step 4 — Define normalization and segmentation helpers

`normalize_image` performs per-region intensity normalization before the model runs, so Cellpose's built-in normalization can be disabled for deterministic behavior. `segment_cells` builds the 3-channel composite Cellpose expects and runs both the cell and nuclear models on a single tile.


In [ ]:
def normalize_image(image, region_size=1824):
    """
    Normalize each tile region independently before passing the image
    to the segmentation model. Performed manually so that Cellpose
    normalization is disabled during model inference.
    """
    image_norm = np.zeros_like(image, np.single)

    for xi in range(int(image.shape[1] / region_size)):
        for yi in range(int(image.shape[0] / region_size)):
            cropped = image[
                yi * region_size:(yi + 1) * region_size,
                xi * region_size:(xi + 1) * region_size
            ]
            cropped = transforms.normalize_img(
                cropped.reshape(cropped.shape[0], cropped.shape[1], 1)
            ).reshape(cropped.shape[0], cropped.shape[1])
            image_norm[
                yi * region_size:(yi + 1) * region_size,
                xi * region_size:(xi + 1) * region_size
            ] = cropped

    return image_norm


def segment_cells(cell_image, nuclear_image, actin_image,
                  cell_model_path, nuclear_model_path,
                  cell_diameter, use_gpu=False):
    """
    Run cell and nuclear segmentation for a single tile.

    Parameters
    ----------
    cell_image         : Cell-membrane channel image array
    nuclear_image      : Nucleus channel image array
    actin_image        : Actin channel image array. Pass None for
                         Cell Paint only (2-channel) runs.
    cell_model_path    : Full path to the cell segmentation model file
    nuclear_model_path : Full path to the nuclear segmentation model file
    cell_diameter      : Diameter in pixels. Used only for the General
                         model. Pass 0 to use the model's internal
                         training diameter.
    use_gpu            : Set to True if a GPU is available
    """
    cell_image    = normalize_image(cell_image)
    nuclear_image = normalize_image(nuclear_image)

    # Build a 3-channel composite: [cell-membrane, nucleus, actin].
    # For Cell Paint only runs, actin_image is None and channel 2
    # remains zeros. Use a _2ch model in that case.
    composite = np.zeros((cell_image.shape[0], cell_image.shape[1], 3))
    composite[:, :, 0] = cell_image
    composite[:, :, 1] = nuclear_image

    if actin_image is not None:
        actin_image = normalize_image(actin_image)
        composite[:, :, 2] = actin_image

    print(f"Using {cell_model_path} for cell segmentation")

    # Use pretrained_model for Element models. model_type is reserved
    # for built-in Cellpose models such as cyto3 and does not load
    # local files correctly.
    model = models.CellposeModel(
        gpu=use_gpu,
        pretrained_model=cell_model_path,
        nchan=3,
    )

    cell_mask, _, _ = model.eval(
        composite,
        diameter=cell_diameter if cell_diameter > 0 else None,
        channels=None,
        normalize=False,   # normalization handled manually above
        resample=False,
    )
    cell_mask = cell_mask.astype(np.uint32)

    print(f"Using {nuclear_model_path} for nuclear segmentation")

    nuclear_model_obj = models.CellposeModel(
        gpu=use_gpu,
        pretrained_model=nuclear_model_path,
    )

    nuclear_mask, _, _ = nuclear_model_obj.eval(
        nuclear_image,
        normalize=False,
        resample=False,
    )

    binary_nuclei = nuclear_mask.copy()
    binary_nuclei[nuclear_mask > 0] = 1

    return cell_mask, binary_nuclei


## Step 5 — Build the tile list from `RunParameters.json`

`RunParameters.json` enumerates every tile imaged on the run and the well each tile belongs to. This cell flattens that into a `tiles` list and a `tile2well` lookup the segmentation loop uses next.


In [ ]:
# Read RunParameters.json to build the full list of tiles and map
# each tile back to its well location.
with open(os.path.join(run_directory, "RunParameters.json")) as f:
    run_parameters = json.load(f)

tile2well = {}
tiles = []

for well in run_parameters["Wells"]:
    for tile in well["Tiles"]:
        tile2well[tile["Name"]] = well["WellLocation"]
        tiles.append(tile["Name"])

print(f"Total tiles to process: {len(tiles)}")


## Step 6 — Segment every tile and write masks

For each tile, the loop loads the channel TIFFs, runs `segment_cells`, and writes a `{tile}_Cell.tif` and `{tile}_Nuclear.tif` to the well's output directory. The actin channel is loaded only when present, so the same code path handles Cell Paint only and Teton runs.


In [ ]:
print(f"Beginning segmentation across {len(tiles)} tiles")

for tile in tiles:
    well          = tile2well[tile]
    cell_type     = cell_types.get(well, "OTHER")
    cell_diameter = cell_diameters.get(well, 0)

    cell_model_path    = os.path.join(
        model_dir,
        element_models.get(cell_type.upper(), element_models["OTHER"]),
    )
    nuclear_model_path = os.path.join(model_dir, nuclear_model)

    os.makedirs(
        os.path.join(output_location, f"Well{well}"), exist_ok=True
    )

    cell_image    = skimage.io.imread(
        os.path.join(run_directory, "Projection", f"Well{well}", f"CP01_{tile}_Cell-Membrane.tif")
    )
    nuclear_image = skimage.io.imread(
        os.path.join(run_directory, "Projection", f"Well{well}", f"CP01_{tile}_Nucleus.tif")
    )

    # Load the actin channel only for Teton and Teton Atlas runs.
    actin_path = os.path.join(
        run_directory, "Projection", f"Well{well}", f"CP01_{tile}_Actin.tif"
    )
    actin_image = skimage.io.imread(actin_path) if os.path.exists(actin_path) else None

    cell_mask, binary_nuclei = segment_cells(
        cell_image, nuclear_image, actin_image,
        cell_model_path, nuclear_model_path,
        cell_diameter, use_gpu=use_gpu,
    )

    skimage.io.imsave(
        os.path.join(output_location, f"Well{well}", f"{tile}_Cell.tif"),
        cell_mask.astype(np.uint16),
    )
    skimage.io.imsave(
        os.path.join(output_location, f"Well{well}", f"{tile}_Nuclear.tif"),
        binary_nuclei.astype(np.uint8),
    )

    print(f"Finished segmenting {tile}")


## Reference

### Runtime expectations

Full segmentation processes every tile in the run. Approximate run times are as follows:

| Plate format | Approximate tiles | CPU estimate | GPU estimate |
| ------------ | ----------------- | ------------ | ------------ |
| 1-well       | ~18 tiles         | ~3 hours     | ~20 minutes  |
| 12-well      | ~216 tiles        | ~36 hours    | ~4–8 hours   |
| 48-well      | ~864 tiles        | ~48 hours    | ~16–24 hours |

### Output files

For each tile, the loop writes two files to your `output_location`:

- `{tile}_Cell.tif`: a `uint16` label mask where each unique integer represents one segmented cell.
- `{tile}_Nuclear.tif`: a `uint8` binary mask where `0` indicates no nucleus and `1` indicates a nucleus is present.

After all tiles finish, re-run Cells2Stats with `--segmentation` pointing at `output_location` to regenerate the cell table. See [Re-run Cells2Stats for cell assignment](https://docs.elembio.io/docs/tutorials/cytoprofiling/custom-segmentation/#cell-assignment).
